# Weather Tail-Harvest — Buy-Side Analysis

## Strategy under evaluation

When a weather-market token level-breaks above a barrier `p` (i.e. the token's price was
below `p` and then crosses up through `p`) inside the 24h pre-resolution window, place a
buy at best offer and hold to resolution. The position pays $1 if the token resolves to 1,
$0 if it gets chopped (resolves to 0).

## TL;DR

- Pooled per-signal edge by entry barrier (24h window):

| p | n_crossed/yr | chop rate | edge $/$1 |
|---|---|---|---|
| 0.50 | 8,313 | 49.7% | +0.3¢ |
| 0.60 | 8,938 | 39.2% | +0.8¢ |
| **0.70** | **9,033** | **28.7%** | **+1.3¢** ← argmax |
| 0.80 | 9,045 | 19.6% | +0.4¢ |
| 0.85 | 9,254 | 14.7% | +0.3¢ |
| 0.90 | 9,819 | 10.6% | −0.6¢ |
| 0.95 | 11,320 | 5.5% | −0.5¢ |

- Best pooled edge is **+1.3¢/$1 at p = 0.70**. Does not survive 1¢ of slippage.
- At p ≥ 0.90 the edge is negative — a late-window level-break to 0.90 is more often a
  head-fake than a real signal.
- The strategy in this exact form does not have a workable edge. The rest of the notebook
  decomposes by family / time-to-chop / first-to-cross policy to surface where a
  filtered variant might still work.

## Universe

37,044 weather markets matching `^(will-the-)?(highest|lowest)-temperature-in-` with end
dates in 2025-05-14 → 2026-05-14. 100% NegRisk. Source + pipeline:
`polymarket/research/scripts/weather_tail_analysis.py`.

## Definition: "crossing"

A token is considered to have **crossed** barrier `p` in the 24h window iff:
- its `min_price` in the window is **below `p`**, AND
- its `max_price` in the window is **at least `p`**.

This matches a level-break trigger (the bot would only fire when price moves through `p`,
not when the token is already sitting at $0.95). All chop rates and edges below are
computed on this set.

## What's in this notebook

1. Load aggregate + per-instance parquets.
2. Win-rate curves per family vs barrier `p`.
3. Time-to-chop distribution — how fast does a losing position die?
4. Best entry `p` — optimization under four objectives.
5. First-to-cross policy — recompute when the bot buys at most one token per market.
6. Per-family ranking at the chosen `p` + entry-slippage sensitivity.
7. 24h vs 48h window sensitivity.
8. Recommended operating spec.

## Caveats

- Analysis covers only the 24h window pre-resolution.
- Fees ignored. Polymarket taker fees are nominally zero today.
- Time-to-chop fallback share is small pooled but high (~85%) in Seoul.


## 1. Load aggregate + per-instance parquets

Files under `polymarket/research/data/analysis/`:

| file | contents |
|---|---|
| `weather_tail_analysis.parquet` | per (slug_family, barrier_price) aggregates, 24h window |
| `weather_tail_analysis_48h.parquet` | same schema, 48h sidebar |
| `weather_tail_per_instance.parquet` | one row per (market, token, barrier) — long-form |
| `weather_universe.parquet` | 37k-market catalogue |


In [ ]:
# Cell 1 — load
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

THIS = Path.cwd()
DATA_DIR = (THIS / ".." / "data" / "analysis").resolve()
if not DATA_DIR.exists():
    DATA_DIR = Path("polymarket/research/data/analysis").resolve()

PRIMARY = pd.read_parquet(DATA_DIR / "weather_tail_analysis.parquet")
SIDEBAR = pd.read_parquet(DATA_DIR / "weather_tail_analysis_48h.parquet")
INST    = pd.read_parquet(DATA_DIR / "weather_tail_per_instance.parquet")
UNI     = pd.read_parquet(DATA_DIR / "weather_universe.parquet")

ENTRY_P = 0.80  # strategy candidate; we'll inspect alternatives in §4

# Derived buy-side columns
for df in (PRIMARY, SIDEBAR):
    df["win_rate"]            = 1 - df["chop_rate"]
    df["roi_per_signal_pct"]  = df["edge_per_signal"] / df["barrier_price"] * 100
    df["kelly_fraction"]      = np.where(df["barrier_price"] < 1.0,
                                         df["edge_per_signal"] / (1 - df["barrier_price"]),
                                         np.nan)
    df["dollar_pnl_pooled"]   = df["n_crossed"] * df["edge_per_signal"]

print(f"primary 24h rows: {len(PRIMARY)}")
print(f"sidebar 48h rows: {len(SIDEBAR)}")
print(f"per-instance rows: {len(INST):,}")
print(f"universe markets : {len(UNI)}")
print(f"Universe date range: {UNI['end_ts'].min()} → {UNI['end_ts'].max()}")
nr_true = (UNI["neg_risk"] == True).sum()
print(f"NegRisk=TRUE share: {nr_true}/{len(UNI)} ({nr_true/len(UNI)*100:.1f}%)")

pooled = PRIMARY[PRIMARY["slug_family"] == "POOLED_ALL_WEATHER"].sort_values("barrier_price")
view = pooled[["barrier_price","n_crossed","chop_rate","win_rate",
               "edge_per_signal","roi_per_signal_pct","kelly_fraction","dollar_pnl_pooled"]].copy()
view.columns = ["p","n","chop","win","edge_$/$1","ROI_%","kelly","tot_$"]
print("\nPooled metrics by barrier p (24h):")
print(view.round({"chop":4,"win":4,"edge_$/$1":4,"ROI_%":2,"kelly":4,"tot_$":0}).to_string(index=False))


**Reading this table**

`n_crossed` = number of (market, token) level-break events at each barrier in the last
12 months. Roughly 8–11k signals per barrier — plenty of statistical power.

`chop_rate` = fraction of those events where the token resolved to 0. Falls monotonically
from ~50% at p=0.50 to ~5% at p=0.95 — the market gets more accurate as it gets more
confident.

`edge_$/$1` = `(1 − chop_rate) − p` = expected profit on a $1-notional position. Peaks at
**+1.3¢/$1 at p = 0.70**, near-zero at p = 0.80, negative at p ≥ 0.90. The pooled
strategy is essentially flat: there's no fat tail to harvest at any single barrier.

`tot_$` = `n_crossed × edge` = total expected $ if the bot took every signal at $1/signal.
Peaks at ~$118 at p = 0.70 — i.e. the strategy's whole annual edge across the universe
sums to ~$120 at $1 sizing. Doesn't even survive fees if those reappear.


## 2. Win-rate curves — where buying is profitable

Y-axis: `1 − chop_rate` = probability the position pays $1 given the token level-breaks
at `p`. The dashed `y = p` diagonal is the **buy breakeven**. Curves above the diagonal
mean buy is profitable.

A vertical purple line marks `p = 0.80` (the original strategy candidate); a green line
marks `p = 0.70` (the pooled-edge argmax).


In [ ]:
# Cell 2 — win-rate curves
top_families = (
    PRIMARY[PRIMARY["slug_family"] != "POOLED_ALL_WEATHER"]
    .sort_values("n_total_instances", ascending=False)
    .drop_duplicates("slug_family")
    .head(10)["slug_family"]
    .tolist()
)
pooled_p = PRIMARY[PRIMARY["slug_family"] == "POOLED_ALL_WEATHER"].sort_values("barrier_price")

fig, ax = plt.subplots(figsize=(11, 6))
for fam in top_families:
    sub = PRIMARY[PRIMARY["slug_family"] == fam].sort_values("barrier_price")
    ax.plot(sub["barrier_price"], sub["win_rate"], marker="o", lw=1.2, alpha=0.75, label=fam)
ax.plot(pooled_p["barrier_price"], pooled_p["win_rate"], marker="s",
        color="black", lw=2.5, label="POOLED_ALL_WEATHER")

xs = np.linspace(0.5, 0.95, 50)
ax.plot(xs, xs, "--", color="red", lw=1.0, label="Buy breakeven y = p")
ax.fill_between(xs, xs, 1.0, color="green", alpha=0.05, label="Buy-profitable region")
ax.axvline(0.70,    color="green",  ls=":", lw=1.2, label="pooled-edge argmax p=0.70")
ax.axvline(ENTRY_P, color="purple", ls=":", lw=1.2, label=f"original candidate p={ENTRY_P}")

ax.set_xlabel("Barrier price p (entry)")
ax.set_ylabel("Win rate  P(resolve=1 | crossed p)")
ax.set_title("Highest/lowest-temperature markets — buy-side win rate by barrier (24h pre-resolution)")
ax.set_ylim(0.4, 1.02)
ax.grid(alpha=0.3)
ax.legend(loc="lower right", fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

# Pooled per-signal edge by p
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(pooled_p["barrier_price"], pooled_p["edge_per_signal"], marker="o", lw=2.5, color="#2c3e50")
ax.axhline(0, color="black", lw=0.8)
ax.axvline(0.70,    color="green",  ls=":", lw=1.2, label="argmax p=0.70")
ax.axvline(ENTRY_P, color="purple", ls=":", lw=1.2, label=f"candidate p={ENTRY_P}")
ax.set_xlabel("Barrier p"); ax.set_ylabel("Pooled edge ($/$1)")
ax.set_title("Per-signal edge by p")
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()


**Interpretation**

The pooled curve sits *barely* above the buy-breakeven diagonal in `p ∈ [0.55, 0.85]`,
crosses *below* at `p ≥ 0.90`. The gap is at most ~2 percentage points — far too thin
to fund the strategy after execution costs.

City-level curves vary slightly: London (highest n) tracks the pooled curve closely;
Seoul sits noticeably below the pack (more chops); Wellington and Buenos Aires alternate
above and below. No top-volume family clears 5¢/$1 at any barrier on this pooled view.

**Why the curve crosses below the diagonal at p = 0.90+**: tokens that *level-break*
through 0.90 in the last 24h are a thin slice — the bulk of eventually-winning tokens
were already at 0.90+ when the window opened (those don't count here, by definition).
The subset that *moves through* 0.90 late is a mix of late-breaking-news rallies and
head-fakes; the head-fakes outweigh the truth-rallies, so empirical win rate < 0.90.


## 3. Time-to-chop distribution

For positions that DO get chopped (the minority), how fast does it happen? Two reasons
this matters:

1. **Stop-loss feasibility.** Fast chops (<1h) are essentially unstoppable. Slow chops
   can be cut at, say, p → 0.50 before they bottom out.
2. **Holding-period risk.** Slower chops mean capital is locked in losers longer.

Boxplots at `p = 0.70` (argmax) and `p = 0.80` (original candidate). Orange line = 1h.


In [ ]:
# Cell 3 — time-to-chop
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, p in zip(axes, [0.70, 0.80]):
    boxes, labels = [], []
    for fam in top_families + ["POOLED_ALL_WEATHER"]:
        sub = INST[(INST["slug_family"] == fam) & (INST["barrier_price"] == p) & INST["crossed_and_crashed"]]
        vals = sub["time_to_5c_hours"].dropna().values
        if len(vals) < 3: continue
        boxes.append(vals); labels.append(f"{fam}\nn={len(vals)}")
    if boxes:
        ax.boxplot(boxes, tick_labels=labels, showfliers=False)
        ax.axhline(24, color="grey", ls="--", lw=0.7, label="24h window edge")
        ax.axhline(1,  color="orange", ls="--", lw=0.7, label="1h reaction floor")
    ax.set_title(f"Time-to-chop at p={p:.2f}  (chopped subset)")
    ax.set_ylabel("hours from cross @ p → first fill ≤ $0.05")
    ax.tick_params(axis="x", rotation=60, labelsize=7)
    ax.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()

rows = []
for p in [0.70, 0.80, 0.85]:
    for fam in top_families + ["POOLED_ALL_WEATHER"]:
        sub = INST[(INST["slug_family"] == fam) & (INST["barrier_price"] == p) & INST["crossed_and_crashed"]]
        if len(sub) == 0: continue
        rows.append({"p": p, "slug_family": fam, "n_chopped": len(sub),
                     "p10_hr":   sub["time_to_5c_hours"].quantile(0.10),
                     "median_hr": sub["time_to_5c_hours"].median(),
                     "p90_hr":   sub["time_to_5c_hours"].quantile(0.90),
                     "pct_filled_to_5c": (sub["t5c_source"] == "fill_below_5c").mean()})
print("Time-to-chop quantiles (chopped subset, per family):")
print(pd.DataFrame(rows).round(2).to_string(index=False))

print("\nPooled chop-speed bins:")
flash_rows = []
for p in [0.70, 0.80, 0.85, 0.90]:
    pooled_sub = INST[(INST["barrier_price"] == p) & INST["crossed_and_crashed"]]
    if len(pooled_sub) == 0: continue
    t = pooled_sub["time_to_5c_hours"].dropna()
    flash_rows.append({
        "p": p, "n_chopped": len(t),
        "share_<1h":   (t < 1).mean(),
        "share_1-6h":  ((t >= 1) & (t < 6)).mean(),
        "share_6-12h": ((t >= 6) & (t < 12)).mean(),
        "share_>=12h": (t >= 12).mean(),
    })
print(pd.DataFrame(flash_rows).round(3).to_string(index=False))


**Interpretation**

Pooled median time-to-chop is ~10–16h at p = 0.70 / 0.80 — most chops play out over
most of the trading day pre-resolution. A stop-loss is mechanically feasible. ~10–20%
of chops finish inside the first hour ("flash chops") — those are unstoppable.

Seoul has noticeably faster chops (median ~5–8h) AND a high fallback share (~85%),
meaning the price doesn't decay visibly through the book — the market just resolves.
Seoul is harder to stop-loss; consider excluding it.

Given the pooled edge is essentially zero, a stop-loss isn't a path to alpha here — it
would just be variance reduction on a near-zero-EV strategy.


## 4. Best entry `p` — four objectives

The four objectives:

| objective | what it favours |
|---|---|
| per-signal $ edge `(win − p)` | barrier where the gap to breakeven is widest |
| ROI per signal `edge / p` | barrier where return per $ tied up is highest |
| Kelly fraction `edge / (1−p)` | growth-rate-maximising barrier (asymmetric loss aware) |
| total $ over universe `n × edge` | barrier where annual $ profit at fixed sizing peaks |


In [ ]:
# Cell 4 — best entry p
pooled = PRIMARY[PRIMARY["slug_family"] == "POOLED_ALL_WEATHER"].sort_values("barrier_price").copy()
show = pooled[["barrier_price","n_crossed","chop_rate","win_rate",
               "edge_per_signal","roi_per_signal_pct","kelly_fraction","dollar_pnl_pooled"]].copy()
show.columns = ["p","n","chop","win","edge_$/$1","ROI_%","kelly","tot_$"]
print("Pooled metrics by barrier p:")
print(show.round({"chop":4,"win":4,"edge_$/$1":4,"ROI_%":2,"kelly":4,"tot_$":0}).to_string(index=False))

print("\nArgmax per objective:")
for col, label in [("edge_per_signal",     "per-signal edge ($/$1)"),
                   ("roi_per_signal_pct",  "ROI per signal (%)"),
                   ("kelly_fraction",      "Kelly fraction"),
                   ("dollar_pnl_pooled",   "total $ over universe")]:
    best = pooled.loc[pooled[col].idxmax()]
    print(f"  {label:25} → p={best['barrier_price']:.2f}  "
          f"(edge={best['edge_per_signal']:+.4f}, win={best['win_rate']:.3f}, "
          f"chop={best['chop_rate']:.3f}, n={int(best['n_crossed'])})")

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
metrics = [
    ("edge_per_signal",    "Per-signal edge ($/$1)",         axes[0,0]),
    ("roi_per_signal_pct", "ROI per signal (%)",             axes[0,1]),
    ("kelly_fraction",     "Kelly fraction f*",              axes[1,0]),
    ("dollar_pnl_pooled",  "Total $ over universe ($1/sig)", axes[1,1]),
]
for col, title, ax in metrics:
    ax.plot(pooled["barrier_price"], pooled[col], marker="o", color="#2c3e50", lw=2)
    best_p = pooled.loc[pooled[col].idxmax(), "barrier_price"]
    ax.axhline(0, color="black", lw=0.6)
    ax.axvline(best_p, color="green", lw=1, ls="--", label=f"argmax p={best_p:.2f}")
    ax.set_xlabel("Barrier p"); ax.set_ylabel(title); ax.set_title(title)
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


**Interpretation**

All four objectives argmax at **p = 0.70** by tiny margins (~+1.3¢/$1 edge, +1.86% ROI,
Kelly ≈ 0.044, ~$118 total $ over the universe). The optimum is real but the absolute
edge is small enough that any 1–2¢ of slippage destroys it.

At `p ≥ 0.90` the edge is **negative** — buying a token that level-breaks through 0.90
in the last 24h is a losing play.


## 5. First-to-cross policy

The bot's policy is "buy the FIRST token in each market (bucket) to cross `p`," not
"buy every token that crosses." At most one trade per market per barrier. Recompute the
chop rate under that constraint, and also decompose the "both-crossed" sub-population
where adverse selection lives.


In [ ]:
# Cell 5 — first-to-cross
sub = INST[INST["crossed"]].sort_values("first_cross_ts")
first = sub.drop_duplicates(["market_id","barrier_price"], keep="first")

rows = []
for p in [0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95]:
    pooled_sub = INST[(INST["barrier_price"] == p) & INST["crossed"]]
    fst_sub    = first[first["barrier_price"] == p]
    rows.append({
        "p": p,
        "n_per_token":   len(pooled_sub),
        "chop_per_token": pooled_sub["crossed_and_crashed"].mean(),
        "n_first_to_cross":    len(fst_sub),
        "chop_first_to_cross": fst_sub["crossed_and_crashed"].mean(),
        "edge_first_to_cross": (1 - fst_sub["crossed_and_crashed"].mean()) - p,
    })
print("Per-token (count each token that crosses)  vs  first-to-cross (one trade per bucket):")
print(pd.DataFrame(rows).round(4).to_string(index=False))

print("\nWithin BOTH-tokens-crossed buckets, who chops more — first or second mover?")
for p in [0.50, 0.70, 0.80, 0.90]:
    cross_p = INST[(INST["barrier_price"] == p) & INST["crossed"]]
    both    = cross_p.groupby("market_id").size()
    both_mkts = both[both == 2].index
    cp = cross_p[cross_p["market_id"].isin(both_mkts)].sort_values("first_cross_ts")
    first_in_both  = cp.drop_duplicates("market_id", keep="first")
    second_in_both = cp.drop_duplicates("market_id", keep="last")
    if len(both_mkts) > 0:
        print(f"  p={p:.2f}: n_both_crossed={len(both_mkts):>5d}  "
              f"first chop={first_in_both['crossed_and_crashed'].mean():.3f}  "
              f"second chop={second_in_both['crossed_and_crashed'].mean():.3f}")


**Interpretation**

First-to-cross is slightly better than per-token at every barrier — at p = 0.80, ~19.1%
vs 19.6%. Marginal improvement; both-crossed buckets are a small fraction of all signals.

The both-crossed decomposition tells a sharper story: at p = 0.80, when *both* tokens of
a bucket level-break through 0.80, the **first-to-cross chops 76%** and the
**second-to-cross chops 24%**. The late mover is the eventual winner. The bot's
"first-to-cross" policy is adversely selected against in that small sub-population.

**Cheap filter to implement at trigger time**: check the *other* outcome token's
best-bid. If ≥ ~$0.20, the market is flippy/both-crossing — abort. If < ~$0.20, proceed.
This excludes ~3–10% of triggers at p = 0.80 and removes the 76%-chop bucket. But on
the corrected economics, even with this filter the edge remains small.


## 6. Per-family ranking at `p = 0.80` and entry-slippage sensitivity

Two questions:
1. Which families clear breakeven at p = 0.80 (n_crossed ≥ 30)?
2. How much slippage does the pooled edge tolerate at each barrier?


In [ ]:
# Cell 6 — family ranking + slippage
df = PRIMARY[(PRIMARY["barrier_price"] == ENTRY_P) & (PRIMARY["n_crossed"] >= 30)].copy()
df = df[df["slug_family"] != "POOLED_ALL_WEATHER"]
df_show = df.sort_values("edge_per_signal", ascending=False) \
            [["slug_family","n_crossed","n_crossed_and_crashed",
              "win_rate","edge_per_signal","roi_per_signal_pct"]]
df_show.columns = ["slug_family","n","n_chop","win","edge_$/$1","ROI_%"]
print(f"Top 25 families at p={ENTRY_P} (n>=30):")
print(df_show.head(25).round({"win":4,"edge_$/$1":4,"ROI_%":2}).to_string(index=False))

print(f"\nBottom 10 (negative edge):")
print(df_show.tail(10).round({"win":4,"edge_$/$1":4,"ROI_%":2}).to_string(index=False))

# Fraction of families with positive edge by p
print("\nFraction of families with positive edge by p (n>=30):")
counts = []
for p in [0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95]:
    s = PRIMARY[(PRIMARY["barrier_price"] == p) & (PRIMARY["n_crossed"] >= 30) &
                (PRIMARY["slug_family"] != "POOLED_ALL_WEATHER")]
    counts.append({"p": p, "n_families": len(s),
                   "n_pos_edge": int((s["edge_per_signal"] > 0).sum()),
                   "share_pos": (s["edge_per_signal"] > 0).mean()})
print(pd.DataFrame(counts).round(3).to_string(index=False))

# Pooled slippage sensitivity
pooled_p = PRIMARY[PRIMARY["slug_family"] == "POOLED_ALL_WEATHER"].sort_values("barrier_price")
slips = [0.00, 0.01, 0.02, 0.03, 0.05]
M = np.full((len(pooled_p), len(slips)), np.nan)
for i, (_, r) in enumerate(pooled_p.iterrows()):
    for j, s in enumerate(slips):
        M[i, j] = r["win_rate"] - (r["barrier_price"] + s)
fig, ax = plt.subplots(figsize=(7.5, 5))
im = ax.imshow(M, aspect="auto", cmap="RdYlGn", vmin=-0.05, vmax=0.05)
ax.set_xticks(range(len(slips))); ax.set_xticklabels([f"+{int(s*100)}c" for s in slips])
ax.set_yticks(range(len(pooled_p))); ax.set_yticklabels([f"p={p:.2f}" for p in pooled_p["barrier_price"]])
ax.set_xlabel("Entry slippage above p"); ax.set_title("Pooled edge ($/$1) — entry = p + slip")
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        v = M[i,j]
        ax.text(j, i, f"{v:+.3f}", ha="center", va="center", fontsize=9,
                color="black" if abs(v) < 0.03 else "white")
fig.colorbar(im, ax=ax, label="edge $/$1")
plt.tight_layout(); plt.show()


**Interpretation**

At p = 0.80 a handful of small-n families have positive edge (Manila +11.7¢ n=36,
Austin +9.4¢ n=47, LA +6.9¢ n=61) — but those numbers are noisy at that sample size and
none are large enough to fund a focused strategy.

The "fraction of families with positive edge" table is the headline negative result: at
every barrier, only about half the high-volume families have any positive edge at all.
At p ≥ 0.90 fewer than half do.

The slippage heatmap is decisive: **no `p` survives 2¢ of pooled slippage**. The
strategy as specified is not fundable.


## 7. 24h vs 48h window sensitivity

Robustness check: is the (essentially zero) edge a 24h artifact, or stable across
window choice?


In [ ]:
# Cell 7 — 24h vs 48h
joined = PRIMARY.merge(SIDEBAR, on=["slug_family","barrier_price"], how="outer", suffixes=("_24h","_48h"))
pooled_cmp = joined[joined["slug_family"] == "POOLED_ALL_WEATHER"].sort_values("barrier_price")
cmp_table = pooled_cmp[["barrier_price",
                        "n_crossed_24h","chop_rate_24h","edge_per_signal_24h",
                        "n_crossed_48h","chop_rate_48h","edge_per_signal_48h"]].copy()
cmp_table.columns = ["p","n_24h","chop_24h","edge_24h","n_48h","chop_48h","edge_48h"]
print("Pooled 24h vs 48h:")
print(cmp_table.round({"chop_24h":4,"edge_24h":4,"chop_48h":4,"edge_48h":4}).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(pooled_cmp["barrier_price"], pooled_cmp["edge_per_signal_24h"], marker="o", lw=2, label="24h primary")
ax.plot(pooled_cmp["barrier_price"], pooled_cmp["edge_per_signal_48h"], marker="s", lw=2, label="48h sidebar")
ax.axhline(0, color="black", lw=0.8)
ax.axvline(0.70,    color="green",  ls=":", lw=1.2, label="argmax p=0.70")
ax.axvline(ENTRY_P, color="purple", ls=":", lw=1.2, label=f"candidate p={ENTRY_P}")
ax.set_xlabel("Barrier p"); ax.set_ylabel("Pooled edge ($/$1)")
ax.set_title("Buy-side edge — 24h vs 48h window")
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()


**Interpretation**

The two windows agree to within ~0.5¢ at every barrier — both very close to zero.
Widening to 48h doesn't help. The (lack of) edge is robust to the window choice.


## 8. Conclusion

### What the analysis says
- Buying a token that level-breaks through `p` in the 24h pre-resolution window does not
  have meaningful edge at any tested barrier `p ∈ {0.50, …, 0.95}`.
- Best pooled gross edge is **+1.3¢/$1 at p = 0.70**. Does not survive 1¢ of slippage.
- At p ≥ 0.90 the edge is negative — late-window level-breaks to high prices are more
  often head-fakes than truth-rallies.

### Why this strategy fails
By the 24h-pre-resolution mark in a NegRisk weather market, the winning bucket is
usually already known to the market and trading at high prices. The thin slice of tokens
that *level-breaks* up to high `p` in those last 24 hours is dominated by late-news
events that don't always pan out — there's no systematic mispricing to harvest.

### What could plausibly still work — open avenues

1. **Speed-of-cross filter.** A token rallying through `p` in seconds may behave
   differently from one drifting through over hours. Per-instance parquet has the data.
2. **Liquidity / volume filter.** Genuine crossings on high-volume markets may behave
   differently from low-volume markets.
3. **Per-family selection.** A handful of families may have edge above pooled — but
   small-n estimates need walk-forward validation before trading.
4. **Finer barrier grid.** We tested `{0.50, …, 0.95}`; the structure at `p ∈ {0.40,
   0.45}` or `{0.97, 0.99}` is uncharacterised.
5. **Asymmetric exits.** Hold-to-resolution is one policy. Scaling out at `p + 5¢` or
   stopping at `p − 5¢` could shift the economics independently of entry.
6. **Earlier window.** This analysis covers only the last 24h. The dynamics at 2–7 days
   pre-resolution may differ.

### Where the strategy almost certainly does NOT work
- Buy every level-break at p = 0.80, hold to resolution. **Net edge after slip ≈ 0.**
- Buy every level-break at p = 0.90 or p = 0.95. **Negative edge before slip.**
